# Explorando o Supabase do Olist Project

Usa a conexão **dona do banco** (`OLIST_PG_DSN`) -- a `rag_reader`
(`OLIST_PG_READER_DSN`, a mesma que o agente usa via `run_sql`) só
enxerga o schema `raw`, de propósito: ela não tem acesso a `app.*`
(logs internos), senão o próprio chat conseguiria vazar essa informação
pros usuários via `run_sql`. Este notebook roda só na sua máquina, então
usar a conexão dona aqui não amplia a superfície do app em produção.

- `raw.*` -- dados da Olist (orders, customers, order_items, reviews, support_tickets, ...)
- `app.rag_query_log` -- log de retrieval (pergunta, chunks recuperados, scores)
- `app.synthetic_traffic_log` -- tráfego sintético dos agentes (persona, pergunta, resposta, latência)

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine

# le o .env da raiz do repo (um nivel acima de notebooks/) -- nao duplica segredo aqui
load_dotenv(Path.cwd().parent / ".env")

engine = create_engine(os.environ["OLIST_PG_DSN"].strip())

%load_ext sql
%config SqlMagic.displaycon = False
%config SqlMagic.autopandas = True
%sql engine

In [2]:
%%sql
SELECT order_status, COUNT(*) AS n
FROM raw.orders
GROUP BY order_status
ORDER BY n DESC;

8 rows affected.

,order_status,n
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [3]:
%%sql
-- ultimas perguntas e o chunk/score que o retrieval trouxe (pro seu TCC: ver onde o top_score veio baixo)
SELECT created_at, question, top_score, chunks
FROM app.rag_query_log
ORDER BY created_at DESC
LIMIT 20;

20 rows affected.

,created_at,question,top_score,chunks
0,2026-11-25 11:05:49.323935,parcela pagamento cartão valor fatura installm...,0.554707,"[{'kind': 'table', 'name': 'order_payments', '..."
1,2026-11-25 11:05:41.528300,"pedido atrasado, status de entrega, prazo esti...",0.560562,"[{'kind': 'metric', 'name': 'tempo_medio_entre..."
2,2026-11-25 11:05:10.925135,ticket médio de vendas por mês,0.445016,"[{'kind': 'metric', 'name': 'ticket_medio', 't..."
3,2026-11-25 11:00:11.517391,valor de pagamento dos pedidos,0.583906,"[{'kind': 'table', 'name': 'order_payments', '..."
4,2026-11-25 11:00:09.428758,valor total de vendas por mês e quantidade de ...,0.466873,"[{'kind': 'table', 'name': 'order_items', 'tex..."
5,2026-11-25 10:58:12.830254,juros parcelamento cartão de crédito parcelas,0.542432,"[{'kind': 'table', 'name': 'order_payments', '..."
6,2026-11-25 10:57:49.831506,valor total de vendas e quantidade de pedidos ...,0.466898,"[{'kind': 'table', 'name': 'order_items', 'tex..."
7,2026-11-25 10:57:44.938892,prazo de entrega do pedido aprovado,0.552128,"[{'kind': 'metric', 'name': 'tempo_medio_entre..."
8,2026-11-25 10:51:00.320320,juros parcelamento cartão de crédito,0.516808,"[{'kind': 'table', 'name': 'order_payments', '..."
9,2026-11-25 10:50:34.629304,ticket médio de vendas por mês,0.445016,"[{'kind': 'metric', 'name': 'ticket_medio', 't..."


In [4]:
%%sql
-- trafego sintetico dos agentes: sucesso e latencia por persona
SELECT persona,
       COUNT(*) AS perguntas,
       AVG(latency_ms)::int AS latencia_media_ms,
       SUM((NOT success)::int) AS falhas
FROM app.synthetic_traffic_log
GROUP BY persona
ORDER BY perguntas DESC;

4 rows affected.

,persona,perguntas,latencia_media_ms,falhas
0,analista_pagamentos,4,5335,0
1,suporte,4,4440,0
2,cliente_atraso,4,3698,0
3,gestor_vendas,4,16956,0
